# P1 폐렴 3분류 — v6 자동 실험 (for 루프, lr 탐색)

**v6 목적:** 여러 하이퍼파라미터 조합을 **한 번에 순차 실행**하고 각각 S3에 저장.
- 기준: v4 설정 + **batch16**(현재 최고 조합) 고정, **lr만 3개 탐색**
- 실험 목록은 CELL 2의 `experiments` 에서 수정 (조합 추가/삭제 자유)
- 각 실험 = 모델 새로 초기화 → 학습 → test 평가 → S3에 run 폴더로 저장
- 마지막에 **요약표** 출력 + **로컬 다운로드 명령** 출력

### ⚠️ 실행 전
1. GPU 인스턴스(`ml.g4dn.xlarge`) → CELL 0에서 CUDA True
2. `download_data.ipynb` 선행 (로컬 데이터) + `pneumonia_labels.csv` 업로드
3. 3개 × 약 12분 = 총 ~35분 예상. 끝나면 인스턴스 **Stop**

## CELL 0 — 디바이스 확인

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| CUDA:', torch.cuda.is_available())
if device.type != 'cuda':
    print('⚠️ GPU 아님 (여러 실험이라 CPU면 매우 오래 걸림)')

## CELL 1 — 공통 설정 + 데이터 분할 (한 번만, 모든 실험 공유)

In [ ]:
import io, json, time, os, numpy as np, pandas as pd, boto3
from PIL import Image

VERSION = 'v6'
DATA_BUCKET = 'aws-say3-dataset1'
OUT_BUCKET  = 'aws-say3-student-dataset'
S3_DEV_ROOT = 'students/aws-say3-11/P1/model_dev'
IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]
NUM_CLASSES = 3
CLASS_NAMES = ['NORMAL', 'BACTERIAL_PNEUMONIA', 'VIRAL_PNEUMONIA']
SEED = 42
VAL_RATIO = 0.15
LOCAL_DATA = os.path.abspath('./data/chest_xray')
assert os.path.isdir(LOCAL_DATA), f'로컬 데이터 없음: {LOCAL_DATA} → download_data.ipynb 먼저'

np.random.seed(SEED); torch.manual_seed(SEED)
s3 = boto3.client('s3')

# ---- 환자 단위 train/val 분할 (v2~v5와 동일) ----
df = pd.read_csv('pneumonia_labels.csv')
test_df   = df[df['split'] == 'test'].reset_index(drop=True)
train_all = df[df['split'] == 'train'].copy()
def pkey(r):
    pid = str(r['patient_id']) if pd.notna(r['patient_id']) else ''
    return pid if pid else 'N_' + r['filename']
train_all['pkey'] = train_all.apply(pkey, axis=1)
patient_label = train_all.groupby('pkey')['label'].agg(lambda s: s.value_counts().index[0])
rng = np.random.default_rng(SEED)
val_patients = set()
for label in sorted(patient_label.unique()):
    pats = patient_label[patient_label == label].index.tolist(); rng.shuffle(pats)
    val_patients.update(pats[:max(1, int(len(pats) * VAL_RATIO))])
is_val = train_all['pkey'].isin(val_patients)
val_df   = train_all[is_val].drop(columns=['pkey']).reset_index(drop=True)
train_df = train_all[~is_val].drop(columns=['pkey']).reset_index(drop=True)
print('train', len(train_df), '/ val', len(val_df), '/ test', len(test_df))
tr_p = set(train_df['patient_id'].dropna()) - {''}; va_p = set(val_df['patient_id'].dropna()) - {''}
print('[검증] 겹치는 환자:', len(tr_p & va_p), '(0이어야 함)')

## CELL 2 — 실험 목록 정의 ★ 여기만 바꾸면 됨
batch16 고정, lr 3개 탐색. 조합을 추가/삭제하거나 다른 하이퍼파라미터도 넣을 수 있다.

In [ ]:
# 각 dict = 실험 하나. (고정) batch16, wd 1e-3, epochs 13, 증강강화
experiments = [
    {'NOTE': 'batch16_lr1e4', 'BATCH_SIZE': 16, 'LR': 1e-4, 'WEIGHT_DECAY': 1e-3, 'EPOCHS': 13},
    {'NOTE': 'batch16_lr3e4', 'BATCH_SIZE': 16, 'LR': 3e-4, 'WEIGHT_DECAY': 1e-3, 'EPOCHS': 13},
    {'NOTE': 'batch16_lr5e5', 'BATCH_SIZE': 16, 'LR': 5e-5, 'WEIGHT_DECAY': 1e-3, 'EPOCHS': 13},
]
for e in experiments: print(e)

## CELL 3 — 공통 함수 (전처리/데이터셋/학습/평가/저장)

In [ ]:
import torchvision.transforms as T
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(12), T.ColorJitter(brightness=0.2, contrast=0.2),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])

class LocalDS(Dataset):
    def __init__(self, rows, tf):
        self.rows = rows.reset_index(drop=True); self.tf = tf
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows.iloc[i]
        im = Image.open(os.path.join(LOCAL_DATA, r['relative_path'])).convert('RGB')
        return self.tf(im), int(r['label'])

def run_epoch(model, loader, criterion, optimizer, train):
    model.train() if train else model.eval()
    tot, preds, labs = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if train: optimizer.zero_grad()
            out = model(imgs); loss = criterion(out, labels)
            if train: loss.backward(); optimizer.step()
            tot += loss.item() * imgs.size(0)
            preds += out.argmax(1).cpu().tolist(); labs += labels.cpu().tolist()
    return tot/len(loader.dataset), accuracy_score(labs, preds), f1_score(labs, preds, average='macro')

# 클래스 가중치 (train 분포 기준, 모든 실험 공통)
counts = train_df['label'].value_counts().sort_index()
class_weights = torch.tensor((counts.sum()/(NUM_CLASSES*counts)).values, dtype=torch.float32).to(device)
print('클래스 가중치:', class_weights.cpu().numpy().round(3).tolist())

def train_one(exp):
    """실험 하나: 학습 → best(val f1) 저장 → test 평가 → S3 저장. 요약 dict 반환."""
    torch.manual_seed(SEED); np.random.seed(SEED)
    bs, lr, wd, ep, note = exp['BATCH_SIZE'], exp['LR'], exp['WEIGHT_DECAY'], exp['EPOCHS'], exp['NOTE']
    print('\n' + '='*70 + f'\n[실험] {note}  (batch {bs}, lr {lr}, wd {wd}, ep {ep})\n' + '='*70)
    tl = DataLoader(LocalDS(train_df, train_tf), batch_size=bs, shuffle=True, num_workers=4, pin_memory=True)
    vl = DataLoader(LocalDS(val_df, eval_tf),    batch_size=bs, shuffle=False, num_workers=4, pin_memory=True)

    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES); model = model.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=ep)

    best_f1, history = -1, []
    ckpt_file = f'best_{note}.pth'
    for epoch in range(1, ep+1):
        t0 = time.time()
        tr = run_epoch(model, tl, criterion, optimizer, True)
        va = run_epoch(model, vl, criterion, optimizer, False)
        scheduler.step()
        mark = ''
        if va[2] > best_f1:
            best_f1 = va[2]
            torch.save({'model_state': model.state_dict(), 'class_names': CLASS_NAMES,
                        'img_size': IMG_SIZE, 'mean': MEAN, 'std': STD}, ckpt_file)
            mark = ' ← best'
        print(f'  [E{epoch:02d}/{ep}] train f1 {tr[2]:.3f} | val f1 {va[2]:.3f} ({time.time()-t0:.0f}s){mark}')
        history.append({'epoch': epoch, 'train_f1': tr[2], 'val_f1': va[2]})

    # test 평가 (best 모델 로드)
    ck = torch.load(ckpt_file, map_location=device); model.load_state_dict(ck['model_state']); model.eval()
    testl = DataLoader(LocalDS(test_df, eval_tf), batch_size=bs, shuffle=False, num_workers=4)
    preds, labs = [], []
    with torch.no_grad():
        for imgs, labels in testl:
            preds += model(imgs.to(device)).argmax(1).cpu().tolist(); labs += labels.tolist()
    test_acc = accuracy_score(labs, preds); test_f1 = f1_score(labs, preds, average='macro')
    report = classification_report(labs, preds, target_names=CLASS_NAMES, digits=4, output_dict=True)
    cm = confusion_matrix(labs, preds)
    print(f'  → test acc {test_acc:.4f} | test macro-F1 {test_f1:.4f}')

    # S3 저장 (run 폴더)
    run_id = time.strftime('%Y%m%d_%H%M')
    run_name = f'run_{run_id}_{note}'
    prefix = f'{S3_DEV_ROOT}/{VERSION}/{run_name}'
    config_dict = {'version': VERSION, 'note': note, 'run_id': run_id,
                   'split': 'patientwise val resplit', 'img_size': IMG_SIZE,
                   'batch_size': bs, 'epochs': ep, 'lr': lr, 'weight_decay': wd, 'seed': SEED,
                   'model': 'resnet18 (ImageNet pretrained)', 'scheduler': 'CosineAnnealingLR'}
    metrics_dict = {'best_val_f1': round(best_f1,4), 'test_acc': round(test_acc,4),
                    'test_macro_f1': round(test_f1,4), 'test_report': report,
                    'confusion_matrix': cm.tolist(), 'history': history}
    with open('config.json','w') as f: json.dump(config_dict, f, ensure_ascii=False, indent=2)
    with open('metrics.json','w') as f: json.dump(metrics_dict, f, ensure_ascii=False, indent=2)
    s3.upload_file(ckpt_file, OUT_BUCKET, f'{prefix}/best_model.pth')
    s3.upload_file('config.json', OUT_BUCKET, f'{prefix}/config.json')
    s3.upload_file('metrics.json', OUT_BUCKET, f'{prefix}/metrics.json')
    print(f'  → S3 저장: s3://{OUT_BUCKET}/{prefix}/')
    return {'note': note, 'batch': bs, 'lr': lr, 'best_val_f1': round(best_f1,4),
            'test_acc': round(test_acc,4), 'test_macro_f1': round(test_f1,4), 'run_name': run_name}

## CELL 4 — 모든 실험 순차 실행 (for 루프)

In [ ]:
results = []
run_names = []
t_all = time.time()
for exp in experiments:
    r = train_one(exp)
    results.append(r); run_names.append(r['run_name'])
print(f'\n모든 실험 완료. 총 {(time.time()-t_all)/60:.1f}분')

## CELL 5 — 요약표 (어떤 조합이 best인지 한눈에)

In [ ]:
res_df = pd.DataFrame(results).sort_values('test_macro_f1', ascending=False)
print('=== 실험 요약 (test macro-F1 내림차순) ===')
print(res_df[['note','batch','lr','best_val_f1','test_acc','test_macro_f1']].to_string(index=False))
print('\n★ best:', res_df.iloc[0]['note'], '| test macro-F1', res_df.iloc[0]['test_macro_f1'])

## CELL 6 — 로컬 다운로드 명령 (이번 실험들 전부, json만)

In [ ]:
LOCAL_DEST = '/Users/skku_aws3_16/Documents/안재형/PROJECT/train/model_dev/_s3_results'
PROFILE = 'aws-say3-11'
dest = f'{LOCAL_DEST}/{VERSION}'
src  = f's3://{OUT_BUCKET}/{S3_DEV_ROOT}/{VERSION}/'
cmd = f'mkdir -p "{dest}" && aws s3 sync "{src}" "{dest}" --profile {PROFILE} --exclude "*.pth"'
print('아래 명령을 [맥 로컬 터미널]에 붙여넣으면 이번 v6 실험 결과(json)를 모두 받습니다:')
print('='*72)
print(cmd)
print('='*72)